# 20 — Fannie external population drift and early-warning assessment
## Stage 1 — Verify inputs and measure drift
Repeat Notebook 12's drift definitions on the 22 harmonized Fannie predictors. Use separate pooled 2015–2017 references for full sampled originations and primary eligible loans. The legacy minimum-disclosed FICO proxy is a separate monitoring sensitivity, not a model input or a certified equivalent score. Its mapping remains unapproved for model scoring. No model fitting or scoring. Historical publication dates are not verified.

Stage 1 is preserved below for reproducibility. The default skips recalculation and uses the exact reviewed Stage 1 run. Run All to complete Stage 2. Set REBUILD_STAGE1=True only for a separate reproducibility run; Stage 2 still uses the reviewed manifest.

In [ ]:
REBUILD_STAGE1 = False

In [ ]:
if REBUILD_STAGE1:
    from pathlib import Path
    from datetime import datetime, timezone
    import hashlib, json, zipfile
    import numpy as np
    import pandas as pd
    from scipy.stats import wasserstein_distance
    from IPython.display import display

    ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
    FANNIE = ROOT / 'data/processed/Fannie_Mae'
    def sha(p):
        with p.open('rb') as f:
            return hashlib.file_digest(f, 'sha256').hexdigest()
    def reviewed(pattern, digest):
        matches = sorted(p for p in FANNIE.glob(pattern) if sha(p) == digest)
        assert matches, f'Reviewed manifest missing: {pattern}'
        return matches[0]
    p18 = reviewed('notebook_18d_consolidated_v1/final/*/manifest.json', '41125a32393ce4170dbd9ee75399da6b616975298cbb92dd145a6ebad5f24fbc')
    p19 = reviewed('notebook_19_multi_year_model_reliability/final/*/model_evaluation_manifest.json', '5786e3b73578a0abf47766c1fba753984b054a29265fea5d430971c7d9cb26ac')
    m18, m19 = [json.loads(p.read_text()) for p in [p18,p19]]
    DATA = p18.parent / 'fannie_2015_2023_sampled_dataset.parquet'
    assert sha(DATA) == m18['dataset_sha256'] == m19['dataset_sha256']
    assert not m19['models_updated'] and not m19['models_fitted'] and not m19['fico_sensitivity']
    features = m19['predictors']
    assert len(features) == 22 and set(features) == set(m18['primary_features'])
    df = pd.read_parquet(DATA)
    assert len(df)==450000 and df.loan_id.is_unique and df.loan_id.notna().all()
    assert df.groupby('origination_year').size().to_dict()==dict.fromkeys(range(2015,2024),50000)
    assert int(df.primary.sum())==430569 and int(df.TARGET_PRIMARY_COMPOSITE.sum())==7223
    assert pd.api.types.is_bool_dtype(df.primary) and df.primary.notna().all()
    assert df.loc[df.primary,'TARGET_PRIMARY_COMPOSITE'].isin([0,1]).all()
    assert df.loc[~df.primary,'TARGET_PRIMARY_COMPOSITE'].isna().all()
    assert df.ORIGINAL_DTI_MISSING.eq(df.ORIGINAL_DTI.isna().astype(int)).all()
    periods = [f'{y}-H{h}' for y in range(2018,2024) for h in (1,2)]
    dev_periods = [f'{y}-H{h}' for y in range(2015,2018) for h in (1,2)]
    assert set(df.half_year)==set(dev_periods+periods)
    print('Verified reviewed dataset and Notebook 19 manifest.')
    # Validate the saved monitoring proxy, without changing model inputs.
    proxy = 'LEGACY_MIN_FICO_PROXY'
    assert m18['fico_sensitivity_approved'] is False
    assert proxy not in features
    model_features = list(features)
    assert set([proxy, 'ORIGINATION_CLASSIC_FICO', 'ORIGINATION_VANTAGESCORE']).issubset(df.columns)
    assert df[['ORIGINATION_CLASSIC_FICO','ORIGINATION_VANTAGESCORE']].isna().all().all()
    score = pd.to_numeric(df[proxy], errors='raise')
    assert score.dropna().between(300,850).all()
    expected_missing = dict(zip(range(2015,2024), [11,11,27,38,20,11,12,32,59]))
    assert score.isna().groupby(df.origination_year).sum().to_dict() == expected_missing
    proxy_profiles = []
    for population, frame in [('full_sample',df),('eligible',df.loc[df.primary])]:
        for period_kind, column in [('annual','origination_year'),('half_year','half_year')]:
            for period, group in frame.groupby(column):
                a = group[proxy].dropna()
                assert len(a)>0
                proxy_profiles.append(dict(population=population,period_kind=period_kind,period=str(period),n=len(group),observed=len(a),missing=len(group)-len(a),missing_pct=100*(1-len(a)/len(group)),minimum=a.min(),q10=a.quantile(.1),q25=a.quantile(.25),median=a.median(),q75=a.quantile(.75),q90=a.quantile(.9),maximum=a.max(),mean=a.mean(),iqr=a.quantile(.75)-a.quantile(.25),distinct=a.nunique()))
        ref = frame.loc[frame.origination_year.le(2017),proxy].dropna()
        assert ref.quantile(.75)>ref.quantile(.25), 'FICO proxy reference IQR unavailable'
    proxy_profiles = pd.DataFrame(proxy_profiles)
    display(proxy_profiles.query("population == 'full_sample' and period_kind == 'annual'"))
    # These checks establish numerical usability only. Historical score semantics
    # and publication timing remain unverified; review before interpretation.
    features = model_features + [proxy]


## Freeze references and calculate drift
Use right-closed development decile bins with duplicate cuts removed. PSI smoothing adds 0.5 counts per bin (0.1 and 1.0 sensitivity). Numeric missingness is separate; categorical missingness is a distinct level. Binary changes are in percentage points. A constant reference uses change-from-constant rather than PSI. No evaluation outcomes define the reference.

In [ ]:
if REBUILD_STAGE1:
    numeric = [proxy,'MI_PERCENT','ORIGINAL_DTI','ORIGINAL_LTV','ORIGINAL_INTEREST_RATE','LOG_ORIGINAL_UPB','CLTV_MINUS_LTV']
    binary = ['HAS_MORTGAGE_INSURANCE','HAS_SUBORDINATE_FINANCING','HIGH_LTV_GT_80','MI_PERCENT_MISSING','ORIGINAL_DTI_MISSING','ORIGINAL_LTV_MISSING','CLTV_MISSING']
    categorical = [f for f in features if f not in numeric+binary]
    assert len(categorical)==9
    signals = {'ORIGINAL_DTI':'wasserstein_iqr_scaled','ORIGINAL_LTV':'wasserstein_iqr_scaled','CHANNEL':'total_variation','ORIGINAL_DTI_MISSING':'binary_absolute_change_pp',proxy:'wasserstein_iqr_scaled'}
    views = {'full_sample':df, 'eligible':df.loc[df.primary]}
    def observed(s):
        a = pd.to_numeric(s,errors='raise').dropna().to_numpy(float)
        assert np.isfinite(a).all()
        return a

    def categories(s):
        return ('V:'+s.astype('string').str.strip()).fillna('M:').value_counts().to_dict()
    def bins(a,cuts):
        return np.bincount(np.searchsorted(cuts,a,side='left'),minlength=len(cuts)+1)
    def psi(r,c,a):
        r,c = np.asarray(r,float),np.asarray(c,float)
        r=(r+a)/(r.sum()+a*len(r)); c=(c+a)/(c.sum()+a*len(c))
        return float(np.sum((c-r)*np.log(c/r)))
    refs, values = {},{}
    for pop, frame in views.items():
        ref=frame.loc[frame.origination_year.le(2017)]
        assert len(ref)==(150000 if pop=='full_sample' else 142506)
        refs[pop]={}; values[pop]={}
        for f in features:
            if f in numeric:
                a=observed(ref[f]); values[pop][f]=a
                cuts=np.unique(np.quantile(a,np.arange(.1,1,.1))) if len(a) and np.unique(a).size>1 else np.array([])
                refs[pop][f]=dict(type='numeric',n=len(ref),observed_n=len(a),iqr=float(np.quantile(a,.75)-np.quantile(a,.25)) if len(a) else None,cuts=cuts.tolist(),counts=bins(a,cuts).tolist(),constant=float(a[0]) if len(a) and np.unique(a).size==1 else None)
            elif f in binary:
                assert ref[f].isin([0,1]).all()
                refs[pop][f]=dict(type='binary',n=len(ref),mean=float(ref[f].mean()))
            else:
                c=categories(ref[f]); c.setdefault('M:',0)
                refs[pop][f]=dict(type='categorical',n=len(ref),counts={k:int(v) for k,v in c.items()})

    def measure(pop, current, f):
        s=refs[pop][f]; n=len(current); assert n>0
        r=dict(feature=f,type=s['type'],reference_n=s['n'],current_n=n,status='ok')
        details=[]
        if f in numeric:
            a=values[pop][f]; b=observed(current[f]); r.update(reference_observed_n=len(a),current_observed_n=len(b),reference_missing_percentage=100*(1-len(a)/s['n']),current_missing_percentage=100*(1-len(b)/n),reference_iqr=s['iqr'])
            r['missing_signed_change_pp']=r['current_missing_percentage']-r['reference_missing_percentage']
            if not len(a) or not len(b): r['status']='no_observations'
            else:
                r.update(reference_mean=float(a.mean()),current_mean=float(b.mean()),reference_median=float(np.median(a)),current_median=float(np.median(b)),wasserstein_raw=float(wasserstein_distance(a,b)))
                r['wasserstein_iqr_scaled']=r['wasserstein_raw']/s['iqr'] if s['iqr']>0 else np.nan
                if s['constant'] is not None:
                    r['status']='reference_constant'; r['constant_changed_percentage']=100*float(np.mean(b!=s['constant']))
                else:
                    c=bins(b,s['cuts']); assert c.sum()==len(b)
                    for v in [.1,.5,1.]: r[f'psi_{v}']=psi(s['counts'],c,v)
                    r['psi']=r['psi_0.5']
                    for i,(x,y) in enumerate(zip(s['counts'],c)):
                        details.append(dict(level_or_bin=str(i+1),reference_count=x,current_count=int(y)))
        elif f in binary:
            assert current[f].isin([0,1]).all()
            r.update(reference_proportion_one=s['mean'],current_proportion_one=float(current[f].mean()))
            r['binary_signed_change_pp']=100*(r['current_proportion_one']-s['mean']); r['binary_absolute_change_pp']=abs(r['binary_signed_change_pp'])
        else:
            c=categories(current[f]); a=s['counts']; levels=sorted(set(a)|set(c))
            r['total_variation']=.5*sum(abs(a.get(k,0)/s['n']-c.get(k,0)/n) for k in levels)
            r['unseen_category_percentage']=100*sum(c.get(k,0) for k in levels if k!='M:' and k not in a)/n
            r.update(reference_missing_percentage=100*a.get('M:',0)/s['n'],current_missing_percentage=100*c.get('M:',0)/n)
            for k in levels: details.append(dict(level_or_bin=k,reference_count=a.get(k,0),current_count=c.get(k,0)))
        return r,details
    rows, distributions=[],[]
    for pop,frame in views.items():
        oot=frame.loc[frame.origination_year.ge(2018)]
        for kind,col in [('half_year','half_year'),('annual','origination_year'),('quarter','ORIGINATION_COHORT')]:
            for period,g in oot.groupby(col,sort=True):
                for f in features:
                    r,d=measure(pop,g,f); key=dict(population=pop,period_kind=kind,period=str(period),feature=f)
                    rows.append(dict(r,**key)); distributions.extend(dict(x,**key) for x in d)
    drift=pd.DataFrame(rows); distribution=pd.DataFrame(distributions)
    assert len(drift)==2*(12+6+24)*23 and not drift.duplicated(['population','period_kind','period','feature']).any()
    print('Drift comparisons:',len(drift))
    drift['analysis_scope'] = np.where(drift.feature.eq(proxy),'fico_proxy_sensitivity','primary')
    distribution['analysis_scope'] = np.where(distribution.feature.eq(proxy),'fico_proxy_sensitivity','primary')


## Development-only candidate cutoffs
Use the maximum signal value across the six development half-years in the full sample. Unequal half-year sizes are retained; annual sampling is fixed, not half-year sampling. Stop if a selected signal is unavailable.

In [ ]:
if REBUILD_STAGE1:
    dev_rows=[]
    for period in dev_periods:
        g=df.loc[df.half_year.eq(period)]
        for f,m in signals.items():
            r,_=measure('full_sample',g,f)
            dev_rows.append(dict(period=period,feature=f,measure=m,value=r.get(m,np.nan),n=len(g)))
    dev_signals=pd.DataFrame(dev_rows)
    assert len(dev_signals)==30 and np.isfinite(dev_signals.value).all()
    cutoffs=dev_signals.groupby(['feature','measure'],as_index=False).agg(cutoff=('value','max'),development_cohorts=('period','nunique'))
    assert cutoffs.development_cohorts.eq(6).all()
    for table in [dev_signals,cutoffs]:
        table['analysis_scope'] = np.where(table.feature.eq(proxy),'fico_proxy_sensitivity','primary')
    display(cutoffs)
    selected=drift.loc[drift.feature.isin(signals)].copy()
    selected['measure']=selected.feature.map(signals)
    selected['value']=[r[m] for (_,r),m in zip(selected.iterrows(),selected.measure)]
    assert np.isfinite(selected.value).all()
    assert len(selected)==2*42*5
    OUT=FANNIE/'notebook_20_external_drift'/'stage1'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    OUT.mkdir(parents=True,exist_ok=False)
    files={'fico_proxy_profiles.csv':proxy_profiles,'feature_drift.csv':drift,'distribution_counts.csv':distribution,'selected_signals.csv':selected,'development_signals.csv':dev_signals,'candidate_cutoffs.csv':cutoffs}
    for name,frame in files.items(): frame.to_csv(OUT/name,index=False)
    (OUT/'frozen_reference_specifications.json').write_text(json.dumps(refs,indent=2,allow_nan=False))
    assert sha(DATA)==m18['dataset_sha256']
    manifest=dict(stage='20_stage1',status='drift_complete_pending_review',dataset_sha256=sha(DATA),notebook19_manifest_sha256=sha(p19),features=model_features,monitoring_features=features,signals=signals,reference_years=[2015,2016,2017],population_views=list(views),cutoff_rule='strictly greater than maximum of six development half-years',historical_predictor_availability='unverified',fico_excluded_from_models=True,fico_proxy_monitoring_sensitivity=True,fico_proxy_semantic_equivalence='unverified',fico_proxy_definition='minimum valid disclosed borrower/co-borrower score from earliest reporting snapshot',fico_proxy_review='numerical checks passed; semantic and temporal interpretation pending review',output_sha256={p.name:sha(p) for p in OUT.iterdir() if p.is_file()})
    (OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
    review=OUT/'notebook_20_stage1_review.zip'
    with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUT.iterdir():
            if p.suffix in ['.csv','.json']: z.write(p,p.name)
    with zipfile.ZipFile(review) as z: assert z.testzip() is None
    print('UPLOAD THIS FILE:',review)

## Stage 1 review completed
The reviewed Stage 1 run is retained unchanged. The FICO proxy is numerically usable as a separate monitoring sensitivity; semantic equivalence and historical availability remain unverified.

## Stage 2 — Load reviewed aggregate results
Use the exact reviewed manifests and verify every aggregate input used below. No models are fitted or rescored. The primary evaluation uses eligible loans; the full sampled population supplies alert signals.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = (Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()).resolve()
FANNIE = ROOT / 'data/processed/Fannie_Mae'
def sha(p):
    with p.open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()
def reviewed(pattern, digest):
    matches = sorted(p for p in FANNIE.glob(pattern) if sha(p)==digest)
    assert matches, f'Reviewed manifest not found: {pattern}'
    return matches[0]
p18=reviewed('notebook_18d_consolidated_v1/final/*/manifest.json','41125a32393ce4170dbd9ee75399da6b616975298cbb92dd145a6ebad5f24fbc')
p19=reviewed('notebook_19_multi_year_model_reliability/final/*/model_evaluation_manifest.json','5786e3b73578a0abf47766c1fba753984b054a29265fea5d430971c7d9cb26ac')
p20=reviewed('notebook_20_external_drift/stage1/*/manifest.json','d604f776ae4661221e853aa8d9d28eef157a854963c648c71f63574b18b9ecad')
m18,m19,m20=[json.loads(p.read_text()) for p in [p18,p19,p20]]
DATA=p18.parent/'fannie_2015_2023_sampled_dataset.parquet'
assert sha(DATA)==m18['dataset_sha256']==m19['dataset_sha256']==m20['dataset_sha256']
assert m20['notebook19_manifest_sha256']==sha(p19) and not m19['models_updated'] and not m19['models_fitted']
protected={p:sha(p) for p in [p18,p19,p20,DATA]}
def read_verified(path, manifest, filename):
    p=path.parent/filename
    assert sha(p)==manifest['output_sha256'][filename], f'Input changed: {filename}'
    protected[p]=sha(p)
    return pd.read_csv(p,dtype={'period':str},float_precision='round_trip')
selected=read_verified(p20,m20,'selected_signals.csv')
cutoffs=read_verified(p20,m20,'candidate_cutoffs.csv')
dev=read_verified(p20,m20,'development_signals.csv')
perf=read_verified(p19,m19,'fannie_performance.csv')
draws=read_verified(p19,m19,'fannie_bootstrap_metric_draws.csv')
cross=read_verified(p19,m19,'cross_agency_performance.csv')
assert len(selected)==420 and len(cutoffs)==5 and len(dev)==30
assert not selected.duplicated(['population','period_kind','period','feature']).any()
assert not cutoffs.duplicated(['feature','measure']).any()
assert dev.groupby(['feature','measure']).period.nunique().eq(6).all()
check=cutoffs.merge(dev.groupby(['feature','measure'],as_index=False).value.max(),on=['feature','measure'],validate='one_to_one')
assert np.allclose(check.cutoff,check.value,rtol=0,atol=1e-15)
assert np.isfinite(selected.value).all()
periods=[f'{y}-H{h}' for y in range(2018,2024) for h in [1,2]]
models=['primary_22__logistic','primary_22__xgboost']
metrics=['roc_auc','average_precision','brier','absolute_risk_gap']
primary=perf.loc[perf.eligibility.eq('primary')].copy()
half=primary.loc[primary.period_kind.eq('half_year')].copy()
assert len(half)==24 and set(half.period)==set(periods) and set(half.model)==set(models)
assert not half.duplicated(['period','model']).any()
assert len(draws)==48000 and not draws.duplicated(['population','model','draw_index']).any()
assert np.isfinite(draws[metrics]).all().all()
assert draws.groupby(['population','model']).draw_index.nunique().eq(1000).all()
assert half.drop_duplicates('period').n.sum()==288063 and half.drop_duplicates('period').events.sum()==6279
print('Reviewed inputs verified; no previous computation repeated.')

## Link population drift with reliability
Compare each of the five signals against four reliability measures, keeping each population view and model separate. Spearman correlations are descriptive; overlapping outcome windows and few cohorts prevent treating them as independent tests. Annual and quarterly results are sensitivity only. AP and Brier require prevalence context.

In [ ]:
link=selected.merge(primary,on=['period_kind','period'],suffixes=('_drift','_reliability'),validate='many_to_many')
assert len(link)==840
assert not link.duplicated(['population_drift','period_kind','period','feature','model']).any()
assert link.loc[link.population_drift.eq('eligible'),'current_n'].eq(link.loc[link.population_drift.eq('eligible'),'n']).all()
assoc=[]
for keys,g in link.groupby(['population_drift','period_kind','feature','analysis_scope','model']):
    for metric in metrics+['prevalence']:
        v=g[['value',metric]].dropna()
        rho=v.value.corr(v[metric],method='spearman') if len(v)>1 and v.value.nunique()>1 and v[metric].nunique()>1 else np.nan
        assoc.append(dict(zip(['population','period_kind','feature','analysis_scope','model'],keys),metric=metric,n_periods=len(v),spearman_rho=rho))
associations=pd.DataFrame(assoc)
assert len(associations)==300
print('Descriptive associations calculated for all resolutions.')

## Reliability changes and candidate deterioration sizes
Use 2018-H1 as a descriptive anchor. Subtract independently resampled cohort and reference metrics using the existing 1,000 draws; reuse the same reference draw index across comparisons. Positive worsening means lower ROC-AUC/AP or higher Brier/absolute gap. Intervals are pointwise, conditional on the frozen models, not adjusted for common shocks. Retain all candidate sizes from Notebook 12.

In [ ]:
thresholds={'roc_auc':[.01,.03,.05],'average_precision':[.005,.01,.02],'brier':[.001,.0025,.005],'absolute_risk_gap':[.0025,.005,.01]}
changes=[]
for model in models:
    p=half.loc[half.model.eq(model)].set_index('period')
    def get_draw(period):
        d=draws.loc[draws.population.eq(f'Fannie {period} primary') & draws.model.eq(model)].sort_values('draw_index')
        assert np.array_equal(d.draw_index,np.arange(1000))
        return d[metrics].to_numpy()
    ref=get_draw(periods[0])
    for period in periods[1:]:
        cur=get_draw(period)
        for j,metric in enumerate(metrics):
            direction=-1 if metric in ['roc_auc','average_precision'] else 1
            delta=direction*(cur[:,j]-ref[:,j])
            lo,hi=np.quantile(delta,[.025,.975])
            changes.append(dict(period=period,reference_period=periods[0],model=model,metric=metric,reference_value=p.loc[periods[0],metric],current_value=p.loc[period,metric],worsening_change=direction*(p.loc[period,metric]-p.loc[periods[0],metric]),worsening_ci_low=lo,worsening_ci_high=hi,valid_draws=len(delta),n=p.loc[period,'n'],events=p.loc[period,'events'],prevalence=p.loc[period,'prevalence']))
changes=pd.DataFrame(changes)
assert len(changes)==88
sensitivity=pd.concat([changes.loc[changes.metric.eq(m)].assign(candidate_worsening_size=t) for m,ts in thresholds.items() for t in ts],ignore_index=True)
sensitivity['deterioration']=sensitivity.worsening_change.gt(sensitivity.candidate_worsening_size)
sensitivity['interval_position']=np.select([sensitivity.worsening_ci_low.gt(sensitivity.candidate_worsening_size),sensitivity.worsening_ci_high.le(sensitivity.candidate_worsening_size)],['entire_interval_above_candidate','entire_interval_at_or_below_candidate'],default='interval_crosses_candidate')
assert len(sensitivity)==264

## Evaluate fixed candidate alerts
Use only full sampled originations and the approved development cutoffs. Classification is provisional and based on point estimates; retain interval uncertainty and compare against always-alert and never-alert benchmarks. The FICO proxy is a separate sensitivity, not a new fitted model or a validated prospective warning.

In [ ]:
alerts=selected.loc[selected.population.eq('full_sample') & selected.period_kind.eq('half_year') & selected.period.isin(periods[1:]),['period','feature','measure','analysis_scope','value']].merge(cutoffs[['feature','measure','cutoff']],on=['feature','measure'],validate='many_to_one')
alerts['candidate_alert']=alerts.value.gt(alerts.cutoff)
assert len(alerts)==55
outcomes=alerts.merge(sensitivity,on='period',validate='many_to_many')
a=outcomes.candidate_alert; d=outcomes.deterioration
outcomes['classification']=np.select([a&d,a&~d,~a&d,~a&~d],['hit','false_alarm','miss','correct_non_alert'],default='invalid')
outcomes['uncertain_outcome']=outcomes.interval_position.eq('interval_crosses_candidate')
assert len(outcomes)==1320 and not outcomes.duplicated(['period','feature','model','metric','candidate_worsening_size']).any()
summary=outcomes.groupby(['feature','analysis_scope','model','metric','candidate_worsening_size'],as_index=False).agg(cohorts=('period','size'),alerts=('candidate_alert','sum'),deteriorations=('deterioration','sum'),hits=('classification',lambda s:s.eq('hit').sum()),false_alarms=('classification',lambda s:s.eq('false_alarm').sum()),misses=('classification',lambda s:s.eq('miss').sum()),correct_non_alerts=('classification',lambda s:s.eq('correct_non_alert').sum()),uncertain_outcomes=('uncertain_outcome','sum'))
assert len(summary)==120 and summary.cohorts.eq(11).all()
assert summary[['hits','false_alarms','misses','correct_non_alerts']].sum(axis=1).eq(11).all()
summary['precision']=summary.hits/summary.alerts.replace(0,np.nan)
summary['recall']=summary.hits/summary.deteriorations.replace(0,np.nan)
summary['false_alarm_rate']=summary.false_alarms/(summary.false_alarms+summary.correct_non_alerts).replace(0,np.nan)
summary['always_alert_hits']=summary.deteriorations
summary['always_alert_false_alarms']=11-summary.deteriorations
summary['never_alert_misses']=summary.deteriorations
freq=alerts.groupby(['feature','analysis_scope'],as_index=False).candidate_alert.sum().rename(columns={'candidate_alert':'fannie_alerts'})
# Reviewed Notebook 12 Stage 5G interpretation, original Freddie 29-feature study.
freddie_counts={'ORIGINAL_DTI':8,'ORIGINAL_LTV':10,'CHANNEL':10,'ORIGINAL_DTI_MISSING':11,'LEGACY_MIN_FICO_PROXY':4}
freq['freddie_alerts']=freq.feature.map(freddie_counts)
freq['freddie_reference']='Notebook 12 Stage 5G reviewed interpretation; original Freddie score for FICO'
display(freq)
display(summary.loc[summary.model.eq('primary_22__xgboost') & summary.metric.eq('brier') & summary.candidate_worsening_size.eq(.0025)])

## Timing feasibility
The earliest saved reporting month is not a historical release date. For each cohort compare the nominal month after cohort end with a hypothetical date waiting for all first reports and the original Freddie October 2022 label-readiness proxy (Notebook 12; not a verified deployment date). Use first payment plus 24 months as the conservative full-window maturity proxy, ignoring early termination and actual publication lags. Report missing timing fields and the fraction of eligible labels with full windows elapsed. Do not interpret these fractions as proven information availability or warning lead time.

In [ ]:
timing_data=pd.read_parquet(DATA,columns=['half_year','primary','report','first_payment'])
assert len(timing_data)==450000
def month_num(s):
    d=pd.to_datetime(s.astype('string'),format='%m%Y',errors='coerce')
    return d.dt.year*12+d.dt.month-1
def fmt(m):
    return f'{int(m)//12:04d}-{int(m)%12+1:02d}' if pd.notna(m) else None
timing_data['report_month']=month_num(timing_data.report)
timing_data['full_window_proxy']=month_num(timing_data.first_payment)+24
model_ready_proxy=2022*12+9
rows=[]
for period in periods:
    g=timing_data.loc[timing_data.half_year.eq(period)]
    e=g.loc[g.primary]
    nominal=int(period[:4])*12+(6 if period.endswith('H1') else 12)
    missing_reports=int(g.report_month.isna().sum())
    last=g.report_month.max()
    aligned=max(nominal,last+1,model_ready_proxy) if not missing_reports else np.nan
    for scenario,date in [('nominal',nominal),('reporting_aligned_proxy',aligned)]:
        known=e.full_window_proxy.notna(); ready=known & e.full_window_proxy.le(date) if pd.notna(date) else pd.Series(False,index=e.index)
        rows.append(dict(period=period,scenario=scenario,decision_month=fmt(date),sampled=len(g),eligible=len(e),first_report_missing=missing_reports,last_first_report=fmt(last),reports_after_or_in_decision=int(g.report_month.ge(date).sum()) if pd.notna(date) else np.nan,model_readiness_proxy_met=bool(date>=model_ready_proxy) if pd.notna(date) else None,maturity_dates_missing=int((~known).sum()),full_window_elapsed=int(ready.sum()) if pd.notna(date) else np.nan,full_window_elapsed_pct=100*ready.sum()/len(e) if pd.notna(date) else np.nan,historical_publication_verified=False))
timing=pd.DataFrame(rows)
assert len(timing)==24
outcomes=outcomes.merge(timing.loc[timing.scenario.eq('reporting_aligned_proxy')],on='period',validate='many_to_one')
display(timing.loc[timing.scenario.eq('reporting_aligned_proxy'),['period','decision_month','full_window_elapsed_pct']])

## Save aggregate review bundle
Retain the complete reviewed inputs and document all limitations. No loan-level records leave the project. The Freddie alert counts are historical Notebook 12 results, not a controlled same-predictor comparison; the companion reliability comparison from Notebook 19 uses identical frozen 22-feature models.

In [ ]:
OUT=FANNIE/'notebook_20_external_drift'/'final'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT.mkdir(parents=True,exist_ok=False)
# Retain matched companion reliability context, not a re-evaluation of Freddie alerts.
context=cross.loc[cross.period_kind.eq('half_year') & ((cross.agency.eq('Freddie') & cross.eligibility.eq('original')) | (cross.agency.eq('Fannie') & cross.eligibility.eq('primary')))].copy()
assert len(context)==48
exports={'drift_reliability_link.csv':link,'spearman_associations.csv':associations,'reliability_changes.csv':changes,'deterioration_sensitivity.csv':sensitivity,'candidate_alerts.csv':alerts,'candidate_alert_evaluation_details.csv':outcomes,'candidate_alert_evaluation_summary.csv':summary,'cross_agency_alert_frequency.csv':freq,'timing_feasibility.csv':timing,'cross_agency_companion_reliability.csv':context}
for filename,frame in exports.items(): frame.to_csv(OUT/filename,index=False)
fig,ax=plt.subplots(figsize=(10,4))
x=np.arange(len(freq)); ax.bar(x-.2,freq.freddie_alerts,.4,label='Freddie (Notebook 12)'); ax.bar(x+.2,freq.fannie_alerts,.4,label='Fannie (Notebook 20)'); ax.set_xticks(x, ['Channel' if f=='CHANNEL' else 'FICO proxy / FICO' if f=='LEGACY_MIN_FICO_PROXY' else f.replace('ORIGINAL_','').replace('_',' ') for f in freq.feature],rotation=20,ha='right'); ax.set_ylim(0,12); ax.set_ylabel('Alerted cohorts (out of 11)'); ax.legend(); fig.tight_layout(); fig.savefig(OUT/'cross_agency_alert_frequency.png',dpi=180); plt.show(); plt.close(fig)
assert all(sha(p)==digest for p,digest in protected.items()), 'A reviewed input changed during evaluation'
manifest=dict(stage='20_final',status='evaluation_complete_pending_interpretation',input_sha256={str(p.relative_to(ROOT)).replace(chr(92),'/'):h for p,h in protected.items()},anchor='2018-H1',candidate_deterioration_sizes=thresholds,cutoff_rule=m20['cutoff_rule'],signals=m20['signals'],fico_proxy_sensitivity=True,models_fitted=False,models_rescored=False,bootstrap_draws_reused=1000,historical_predictor_availability='unverified',timing_assumption='earliest report + 1 month, full window first payment + 24 months; October 2022 Freddie label-readiness proxy, not actual deployment',limitations=['Retrospective exploratory analysis; no validated lead time','Only 11 later half-year cohorts, overlapping outcome windows','Marginal bootstrap intervals conditional on frozen models','AP and Brier depend on prevalence','Development reference overlaps cutoff cohorts','FICO semantics unverified','Freddie alert counts refer to original Notebook 12 models and monitoring population; not a controlled same-feature comparison'],output_sha256={p.name:sha(p) for p in OUT.iterdir() if p.is_file()})
(OUT/'manifest.json').write_text(json.dumps(manifest,indent=2))
review=OUT/'notebook_20_final_review.zip'
with zipfile.ZipFile(review,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        if p.suffix in ['.csv','.json','.png']: z.write(p,p.name)
with zipfile.ZipFile(review) as z: assert z.testzip() is None
print('UPLOAD THIS FILE:',review)